# 🎬 Channel Micro-POV — 1-Click Cloud Render Engine (Google Colab T4 GPU)

**Prinsip Zero-Manual Workflow:** Tidak perlu laptop kencang dan tidak perlu unduh aset manual.
Semua aset CC0 (tekstur PBR, HDRI, model 3D, SFX, BGM), voiceover Gemini 3.8 TTS, render 3D Blender Cycles GPU, subtitle dinamis `.ass`, hingga mixing final `-14 LUFS` berjalan otomatis dan tersimpan permanen di **Google Drive** kamu.

### Persiapan 30 Detik Sebelum Klik *Run All*:
1. Pastikan Runtime memakai GPU: menu **Runtime → Change runtime type → T4 GPU**.
2. Buka ikon kunci **Secrets** di panel kiri Colab → tambahkan secret bernama `GEMINI_API_KEY` (isi API Key Google AI Studio kamu) dan aktifkan toggle **Notebook access**.
3. Pilih mode di sel konfigurasi di bawah (`preview` untuk QC super cepat 540×960, atau `final` untuk 1080×1920 siap upload Shorts/TikTok), lalu klik **Runtime → Run All**!

In [ ]:
#@title 1. Konfigurasi Episode & Mount Google Drive { run: "auto" }
import os
from pathlib import Path

REPO_URL = "https://github.com/ruangcermat/channel-micro-pov.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
EPISODE_SPEC = "04_episodes/ep001_kecoak_dapur/ep001_spec.json" #@param {type:"string"}
RENDER_MODE = "preview" #@param ["preview", "final"]
SINGLE_SCENE = 0 #@param {type:"integer"}
TTS_VOICE = "Puck" #@param ["Puck", "Charon", "Fenrir", "Kore", "Aoede", "Enceladus"]
SKIP_EXISTING_VO = True #@param {type:"boolean"}
EXPORT_PROCEDURAL_GLB_CACHE = False #@param {type:"boolean"}

# 1. Mount Google Drive untuk cache aset & penyimpanan output permanen
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/channel-micro-pov")
ASSETS_DIR = DRIVE_ROOT / "02_assets"
EPISODE_SLUG = Path(EPISODE_SPEC).parent.name
WORK_DIR = DRIVE_ROOT / "04_episodes" / EPISODE_SLUG

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# 2. Ambil GEMINI_API_KEY secara aman dari Colab Secrets
try:
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key.strip()
        print("✅ GEMINI_API_KEY berhasil dimuat dari Colab Secrets.")
    else:
        print("⚠️ Secret GEMINI_API_KEY kosong — isi dulu di panel Secrets Colab!")
except Exception as exc:
    print(f"⚠️ Tidak bisa membaca Colab Secret GEMINI_API_KEY: {exc}")

os.environ["MICRO_POV_ASSETS_DIR"] = str(ASSETS_DIR)
os.environ["MICRO_POV_WORK_DIR"] = str(WORK_DIR)

print(f"📁 Drive Assets Cache : {ASSETS_DIR}")
print(f"📁 Episode Output Dir : {WORK_DIR}")
print(f"🎞️ Render Mode        : {RENDER_MODE} | Single Scene: {SINGLE_SCENE or 'ALL'}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "⚠️ GPU tidak terdeteksi — pastikan Runtime diset ke T4 GPU!"

In [ ]:
#@title 2. Sync Repository GitHub & Install Dependensi Pipeline
import subprocess
import sys

def run_step(cmd):
    """Jalankan subprocess dengan output real-time ke cell Colab (stdout + stderr)."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

REPO_DIR = Path("/content/channel-micro-pov")
if (REPO_DIR / ".git").exists():
    print("🔄 Pulling update terbaru dari GitHub...")
    run_step(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    run_step(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_step(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    print("📥 Cloning repository dari GitHub...")
    run_step(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libglu1-mesa > /dev/null
!pip install -q --upgrade -r /content/channel-micro-pov/03_pipeline/requirements.txt
print("✅ Repo & dependensi Python/FFmpeg/Font siap!")

In [ ]:
#@title 3. Install Blender 4.2 LTS Linux Headless (Cycles GPU + OIDN)
BLENDER_VERSION = "4.2.8"
BLENDER_TAR = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL = f"https://download.blender.org/release/Blender4.2/{BLENDER_TAR}"
BLENDER_BIN = Path("/opt/blender/blender")

if not BLENDER_BIN.exists():
    print(f"⬇️ Mengunduh Blender {BLENDER_VERSION} LTS Linux Headless...")
    !wget -q --show-progress "{BLENDER_URL}" -O "/tmp/{BLENDER_TAR}"
    !mkdir -p /opt/blender
    !tar -xf "/tmp/{BLENDER_TAR}" -C /opt/blender --strip-components=1
    !rm -f "/tmp/{BLENDER_TAR}"
else:
    print("✅ Blender headless sudah terpasang di /opt/blender")

!/opt/blender/blender --version

In [ ]:
#@title 4. Auto-Fetch & Cache Aset 3D/Tekstur PBR/HDRI CC0 (Poly Haven API)
SPEC_PATH = REPO_DIR / EPISODE_SPEC

!python3 "{REPO_DIR}/03_pipeline/scripts/fetch_assets.py" \
    --spec "{SPEC_PATH}" \
    --assets-dir "{ASSETS_DIR}"

if EXPORT_PROCEDURAL_GLB_CACHE:
    !/opt/blender/blender --background --python "{REPO_DIR}/03_pipeline/scripts/procedural_props.py" -- \
        --export-glb "{ASSETS_DIR}"

In [ ]:
#@title 5. Audio-First Voiceover — Gemini 3.8 Flash TTS (Strict speech_metadata)
import json
from IPython.display import Audio, display

tts_cmd = [
    "python3",
    str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
    "--spec", str(SPEC_PATH),
    "--work-dir", str(WORK_DIR),
    "--voice", TTS_VOICE,
]
if SKIP_EXISTING_VO:
    tts_cmd.append("--skip-existing")
if SINGLE_SCENE > 0:
    tts_cmd.extend(["--scene", str(SINGLE_SCENE)])

run_step(tts_cmd)

manifest_path = WORK_DIR / "timing_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"\n⏱️ Total durasi Audio-First: {manifest['total_duration_sec']:.2f}s ({manifest['total_frames']} frames @ {manifest['fps']}fps)")
for sc in manifest["scenes"]:
    print(
        f"  Scene {sc['scene_id']:02d}: {sc['start_sec']:5.2f}s - {sc['end_sec']:5.2f}s "
        f"(VO {sc['audio_duration_sec']:.2f}s, {sc['frame_count']}f) | {sc['vo_text'][:52]}..."
    )

first_wav = Path(manifest["scenes"][0]["wav_path"])
if first_wav.exists():
    print("\n🔊 Preview VO Scene 01:")
    display(Audio(filename=str(first_wav)))

In [ ]:
#@title 6. Auto-Prepare SFX & BGM CC0 (Direct Fetch + Punchy Synth Fallback)
!python3 "{REPO_DIR}/03_pipeline/scripts/prepare_audio.py" \
    --spec "{SPEC_PATH}" \
    --assets-dir "{ASSETS_DIR}"

In [ ]:
#@title 7. Blender Headless 3D Render (Cycles GPU + OpenImageDenoise)
render_cmd = [
    str(BLENDER_BIN),
    "--background",
    "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
    "--",
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
    "--save-blend",
]
if SINGLE_SCENE > 0:
    render_cmd.extend(["--scene", str(SINGLE_SCENE)])

print("🚀 Menjalankan render Blender:", " ".join(render_cmd))
run_step(render_cmd)

In [ ]:
#@title 8. Subtitle Dinamis (.ass) & Final Mixing (-14 LUFS + Auto-Ducking)
compose_cmd = [
    "python3",
    str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
]
run_step(compose_cmd)

In [ ]:
#@title 9. Player Preview Video Langsung di Colab & Lokasi File Google Drive
import base64
from IPython.display import HTML, display

ep_id = manifest["episode_id"].lower()
out_name = f"final_{ep_id}.mp4" if RENDER_MODE == "final" else f"preview_{ep_id}.mp4"
video_path = WORK_DIR / out_name

assert video_path.exists(), f"File video tidak ditemukan: {video_path}"
size_mb = video_path.stat().st_size / (1024 * 1024)
print(f"🎉 SELESAI! Video tersimpan permanen di Google Drive:")
print(f"   📂 {video_path} ({size_mb:.2f} MB)")

if size_mb <= 35.0:
    b64 = base64.b64encode(video_path.read_bytes()).decode("ascii")
    html_player = (
        '<div style="display:flex;justify-content:center;padding:12px;background:#111;border-radius:12px;">'
        '<video width="360" height="640" controls playsinline '
        'style="border-radius:10px;box-shadow:0 8px 24px rgba(0,0,0,0.5);">'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        "</video></div>"
    )
    display(HTML(html_player))
else:
    print("ℹ️ Ukuran file > 35 MB; silakan putar langsung dari Google Drive kamu.")